# LLM 성능평가

LLM 성능평가는 모델이 낸 답이 과업의 성공 조건을 얼마나 만족하는지 증거로 확인하는 과정이다. 모델 벤치마크 점수와 실제 서비스의 답변 품질은 같지 않으므로, 과업·사용자·위험에 맞는 평가 설계를 함께 만든다.

정답이 있는 과업의 자동 지표, 표준 벤치마크, LLM-as-a-Judge, RAG 평가를 연결한다. 다음 노트북에서는 같은 원칙을 `lm-eval run`, 심사 프롬프트, RAGAS 코드로 구현한다.


## 평가 설계의 출발점

평가 전에 성공을 정의한다. 번역은 의미 보존과 자연스러움, 요약은 핵심 정보 보존과 압축, 고객 응대는 정책 준수와 문제 해결을 함께 본다.

개발용 데이터로 프롬프트를 고친 뒤에는 별도의 고정 테스트셋으로 다시 측정해 과적합을 막는다. 실제 서비스의 질문과 실패하기 쉬운 경계 사례를 함께 넣어야 한다.


## LLM 벤치마크

벤치마크는 동일한 문제와 채점 규칙으로 모델을 비교하는 표준 시험이다. 객관식 지식·추론 문제는 정확도나 정답 선택지의 로그우도를 계산하기 쉬우므로 모델의 특정 능력을 비교하는 데 유용하다.

일반 벤치마크는 지식·추론·독해처럼 여러 모델에 공통인 능력을 보고, 도메인 벤치마크는 금융·법률·의료 같은 업무 지식을 본다. 얼라인먼트·안전 평가는 지시 준수, 편향, 유해 출력처럼 사람의 요구와 정책에 부합하는지를 별도의 기준으로 확인한다.

벤치마크 점수는 학습 데이터 오염, 언어·도메인 차이, 프롬프트와 few-shot 설정의 영향을 받는다. 서비스 도입 판단에는 표준 벤치마크와 실제 서비스 입력으로 만든 고정 평가셋을 함께 사용한다.


### 공개 모델 카드의 점수 읽기

MMLU, GSM8K 같은 모델 카드 점수는 특정 데이터셋·프롬프트·few-shot 조건에서 얻은 결과이다. 숫자를 비교할 때는 평가 버전, 샘플 수, 채점 방식, 실행 조건이 같은지 먼저 확인한다.

따라서 모델 카드 점수는 후보 모델을 좁히는 출발점이며, 한국어 업무 문서나 실제 RAG 답변의 품질을 보증하지 않는다.


### 한국어 벤치마크

한국어 벤치마크는 한국어 문장 이해와 추론을 비교하는 평가 집합이다. KoBEST는 BoolQ, COPA, WiC, HellaSwag, SentiNeg처럼 문맥을 읽고 선택지나 라벨을 판단하는 과업을 묶는다. KMMLU는 한국어 시험에서 수집한 여러 전문 분야의 객관식 문제로 지식과 추론을 평가한다. KorQuAD는 주어진 한국어 문서에서 질문의 답이 되는 구간을 찾는 기계독해 과업이다.

선택 문제의 높은 점수는 요약, 대화, 근거 제시 능력을 보장하지 않는다. 과업 형식과 서비스 입력을 분리해 해석한다.


## 모델 평가 지표

지표는 무엇을 세는가에 따라 다른 결론을 낸다. 정확도는 전체 사례 중 정답을 맞힌 비율이다. 정밀도는 모델이 양성이라고 예측한 사례 중 실제 양성의 비율이고, 재현율은 실제 양성 중 모델이 찾아낸 비율이다. F1은 정밀도와 재현율의 조화평균이므로 클래스 불균형이 있거나 두 오류를 함께 관리할 때 유용하다.

분류·선택형처럼 정답 라벨이 명확한 과업은 정확도와 F1을 사용할 수 있다. 열린 생성은 같은 뜻을 여러 표현으로 말할 수 있으므로 Exact Match만으로 평가하면 올바른 답도 오답이 될 수 있다. 문자열 겹침, 의미 유사도, 사실성·안전성에 대한 사람 판단을 과업에 맞게 조합한다.

한 지표의 상승은 좋은 신호이지만 충분한 결론은 아니다. 점수가 오른 사례와 실패 사례를 표본으로 읽어 지표가 놓치는 오류를 확인한다.


## LLM 출력은 어떻게 채점하는가

로그우도 기반 채점은 같은 프롬프트 뒤에 각 후보 문자열이 이어질 조건부 로그확률을 계산하고 가장 큰 후보를 고른다. 후보 길이가 다르면 긴 문자열이 불리할 수 있으므로 벤치마크가 원점수와 길이 정규화 점수 중 무엇을 쓰는지도 확인한다.

생성 기반 채점은 모델이 만든 문자열에서 정답을 추출한 뒤 Exact Match, F1 같은 지표로 기준 답과 비교한다. 예를 들어 `C
해설: ...`에서 정규식으로 `C`만 추출할 수 있지만, 이 후처리 규칙 자체가 점수에 영향을 주므로 평가 설정에 함께 기록해야 한다. 열린 생성은 기준 답과의 겹침, 의미 유사도, 근거 충실도처럼 여러 관점을 분리해 측정한다.

사람 평가는 복잡한 품질을 직접 볼 수 있지만 비용과 일관성 관리가 필요하다. 자동 지표와 LLM 심사는 사람 평가 표본으로 상관관계와 편향을 점검하며 보조적으로 사용한다.


## 평가 자동화 도구: lm-eval-harness

`lm-evaluation-harness`는 모델 백엔드, 과업 설정, few-shot 수, 채점 지표를 한 실행 흐름으로 묶는 평가 도구이다. 현재 CLI는 `lm-eval run`으로 평가를 실행하고 `lm-eval ls tasks`로 과업을 찾으며 `lm-eval validate --tasks <task>`로 과업 설정을 점검한다. 기존의 단일 명령 형식도 호환되지만 수업에서는 하위 명령 형식을 사용한다.

task YAML의 `output_type`은 `multiple_choice`, `loglikelihood`, `loglikelihood_rolling`, `generate_until` 중 하나로 모델 요청 방식을 정한다. `doc_to_text`는 프롬프트, `doc_to_target`은 정답, `doc_to_choice`는 선택지를 만들고 `metric_list`는 집계 지표를 정한다.

실습의 흐름은 `모델 지정 → task YAML 선택 → 추론·채점 → results와 sample 로그 저장 → 오류 문항 분석`이다. `--log_samples`로 개별 입력·출력을 남겨야 평균 점수가 왜 변했는지 문항 단위로 추적할 수 있다. 다음 노트북에서 KoBEST와 KMMLU에 이 흐름을 적용한다. 공식 CLI와 task 설정은 [lm-evaluation-harness 문서](https://github.com/EleutherAI/lm-evaluation-harness/blob/main/docs/interface.md)에서 확인할 수 있다.


## LLM-as-a-Judge

LLM-as-a-Judge는 평가 기준과 후보 답변을 심사 모델에 주고 점수·근거·선호를 받는 방식이다. 의미 보존, 문체, 복합 기준처럼 n-gram 겹침만으로 보기 어려운 품질을 넓은 표본에서 비교할 수 있다.

심사 프롬프트에는 평가 축, 점수 척도, 출력 형식, 동점·불확실성 처리 규칙을 명시한다. 심사 모델은 자기 모델의 답을 선호하는 경향, 먼저 놓인 답을 선호하는 위치 편향, 내용보다 긴 답을 선호하는 장문 편향, 비용과 비결정성의 영향을 받을 수 있다. 사람 평가 표본으로 심사 결과와의 일치도를 확인하고, 두 답을 비교할 때는 모델 이름을 가리고 제시 순서를 바꿔 위치 편향을 점검한다.


## 평가 결과로 다음 실험 정하기

좋은 평가는 순위표를 만드는 데서 끝나지 않는다. 낮은 점수가 검색 누락, 근거 없는 생성, 프롬프트 형식, 모델 지식 부족 중 어디에서 왔는지 분해해야 개선 방법을 고를 수 있다.

비교 실험에서는 모델 외의 조건을 고정하고, 데이터셋 버전·명령·모델 ID·seed·지표를 기록한다. 이 기록이 있어야 다음 실행의 점수 차이를 재현 가능한 변화로 해석할 수 있다.


## 생성 품질 지표: BLEU, ROUGE, BERTScore

BLEU는 후보 문장의 여러 n-gram 정밀도를 기하평균하고, 기준보다 지나치게 짧은 후보에 brevity penalty를 적용한다. 원래 말뭉치 단위 기계번역 평가를 위해 설계되었으며, 표현이 다양하면 의미가 맞아도 낮아질 수 있으므로 번역 품질의 전부를 뜻하지 않는다.

ROUGE는 기준 답과 후보의 n-gram 또는 최장 공통 부분열 겹침을 본다. ROUGE-1은 unigram, ROUGE-2는 bigram, ROUGE-L은 최장 공통 부분열로 순서를 어느 정도 반영한다. 이름은 recall-oriented이지만 구현체는 precision·recall·F1을 함께 보고할 수 있으므로 어떤 값을 사용했는지 기록한다. 요약의 핵심 누락을 찾는 데 유용하지만 사실 오류나 문장 품질을 직접 판정하지는 않는다.

BERTScore는 후보와 기준 토큰의 문맥 임베딩 유사도를 정렬해 precision·recall·F1을 계산한다. 동의어나 어순 차이에 더 관대하지만 사용하는 사전학습 모델과 기준 답의 품질에 영향을 받는다. 아래 코드는 ROUGE-1 재현율의 가장 작은 계산 단위를 확인한다.


In [2]:
from collections import Counter

reference_tokens = '서울 대한민국 수도'.split()
candidate_tokens = '대한민국 수도 수도'.split()


print(reference_tokens)

['서울', '대한민국', '수도']


### 과업에 따라 지표를 고르기

BLEU·ROUGE·BERTScore는 기준 답이 있을 때 후보 텍스트를 비교하는 지표이다. 반면 RAGAS는 질문, 검색된 문맥, 생성 답변, 필요하면 기준 답을 분리해 RAG 파이프라인의 어느 단계가 약한지 본다.

아래 규칙표는 하나의 정답이 아니라 첫 선택 기준이다. 실제 평가에서는 사용자 영향, 비용, 안전 요구를 반영한 사람 검토 표본을 추가한다.
